In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
qs = pd.read_csv('/content/drive/MyDrive/Qs World Ranking 2025.csv')
the = pd.read_csv('/content/drive/MyDrive/The World University Ranking 2016-2026.csv')

In [ ]:
qs.head()
qs.info()
qs.describe()
print(qs.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1503 entries, 0 to 1502
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   2025 Rank                       1503 non-null   object 
 1   2024 Rank                       1482 non-null   object 
 2   Institution Name                1503 non-null   object 
 3   Location                        1503 non-null   object 
 4   Location Full                   1503 non-null   object 
 5   Size                            1503 non-null   object 
 6   Academic Reputation             1503 non-null   float64
 7   Employer Reputation             1503 non-null   float64
 8   Faculty Student                 1503 non-null   float64
 9   Citations per Faculty           1503 non-null   float64
 10  International Faculty           1403 non-null   float64
 11  International Students          1445 non-null   float64
 12  International Research Network  15

In [ ]:
the.info()
the.describe()
print(the.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16713 entries, 0 to 16712
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Rank                     16713 non-null  float64
 1   Name                     16713 non-null  object 
 2   Country                  16713 non-null  object 
 3   Student Population       16713 non-null  object 
 4   Students to Staff Ratio  16713 non-null  float64
 5   International Students   16711 non-null  object 
 6   Female to Male Ratio     15953 non-null  object 
 7   Overall Score            16713 non-null  float64
 8   Teaching                 16713 non-null  float64
 9   Research Environment     16713 non-null  float64
 10  Research Quality         16713 non-null  float64
 11  Industry Impact          16713 non-null  float64
 12  International Outlook    16713 non-null  float64
 13  Year                     16713 non-null  int64  
dtypes: float64(8), int64(1

In [ ]:
qs = qs.drop_duplicates()
the = the.drop_duplicates()

In [ ]:
qs.isnull().sum()
the.isnull().sum()

,0
Rank,0
Name,0
Country,0
Student Population,0
Students to Staff Ratio,0
International Students,2
Female to Male Ratio,760
Overall Score,0
Teaching,0
Research Environment,0


In [ ]:
qs['Institution Name'] = qs['Institution Name'].str.strip().str.lower()
the['Name'] = the['Name'].str.strip().str.lower()
qs['Location'] = qs['Location'].str.strip().str.lower()
the['Country'] = the['Country'].str.strip().str.lower()

In [ ]:
merged = pd.merge(qs, the, left_on=['Institution Name', 'Location'], right_on=['Name', 'Country'], how='outer')

In [ ]:
merged.to_csv('/content/drive/MyDrive/university_cleaned.csv', index=False)

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/university_cleaned.csv')

In [ ]:
# Clean 'Student Population' (from THE rankings in df) and 'International Students_y' (from THE rankings in df)
def clean_population_range(s):
    if pd.isna(s):
        return None
    s = str(s).replace(',', '').strip().lower()
    if s == 'n/a':
        return None
    if '-' in s:
        parts = s.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return None
    elif '+' in s:
        try:
            return float(s.replace('+', ''))
        except ValueError:
            return None
    elif '~' in s:
        try:
            return float(s.replace('~', ''))
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

def clean_percentage_range(s):
    if pd.isna(s):
        return None
    s = str(s).replace('%', '').strip().lower()
    if s == 'n/a':
        return None
    if '-' in s:
        parts = s.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

df['Student_Population_Cleaned'] = df['Student Population'].apply(clean_population_range)
df['International_Students_y_Cleaned'] = df['International Students_y'].apply(clean_percentage_range)

df['Global_Rank_Score'] = 100 - df['Rank']   # Using 'Rank' from THE
df['Research_Productivity_Index'] = df['Research Quality'] / df['Students to Staff Ratio']
df['Faculty_Student_Ratio'] = df['Faculty Student']
df['Intl_Student_Percentage'] = df['International_Students_y_Cleaned']

In [ ]:
df[['Student_Population_Cleaned','International_Students_y_Cleaned','Global_Rank_Score','Research_Productivity_Index']].describe()


,Student_Population_Cleaned,International_Students_y_Cleaned,Global_Rank_Score,Research_Productivity_Index
count,1.671300e+04,16703.000000,16713.000000,16713.000000
mean,2.294569e+04,11.232353,-722.676719,3.680942
std,3.268570e+04,12.181542,531.516691,5.249611
min,2.500000e+01,0.000000,-2091.000000,0.017927
25%,9.839000e+03,2.000000,-1107.000000,1.473333
50%,1.741500e+04,7.000000,-660.000000,2.703911
75%,2.886000e+04,16.000000,-280.000000,4.423256
max,1.824383e+06,96.000000,99.000000,248.000000


In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[['Global_Rank_Score','Research_Productivity_Index','Intl_Student_Percentage']] = scaler.fit_transform(
    df[['Global_Rank_Score','Research_Productivity_Index','Intl_Student_Percentage']]
)

In [ ]:
df['Performance_Index'] = (
    0.4*df['Global_Rank_Score'] +
    0.3*df['Research_Productivity_Index'] +
    0.3*df['Intl_Student_Percentage']
)

In [ ]:
import plotly.express as px

fig = px.bar(df.sort_values('Performance_Index',ascending=False).head(10),
             x='Name', y='Performance_Index',
             title='Top 10 Universities by Performance Index')
fig.show()

In [ ]:
import plotly.express as px
df['Performance_Index'] = (
    0.4*df['Global_Rank_Score'] +
    0.3*df['Research_Productivity_Index'] +
    0.3*df['Intl_Student_Percentage']
)

print(df.columns)

fig = px.bar(
    df.sort_values('Performance_Index', ascending=False).head(10),
    x='Name',   # <-- Replaced 'University Name' with 'Name'
    y='Performance_Index',
    title='Top 10 Universities by Performance Index'
)

fig.show()

Index(['2025 Rank', '2024 Rank', 'Institution Name', 'Location',
       'Location Full', 'Size', 'Academic Reputation', 'Employer Reputation',
       'Faculty Student', 'Citations per Faculty', 'International Faculty',
       'International Students_x', 'International Research Network',
       'Employment Outcomes', 'Sustainability', 'QS Overall Score', 'Rank',
       'Name', 'Country', 'Student Population', 'Students to Staff Ratio',
       'International Students_y', 'Female to Male Ratio', 'Overall Score',
       'Teaching', 'Research Environment', 'Research Quality',
       'Industry Impact', 'International Outlook', 'Year',
       'Student_Population_Cleaned', 'International_Students_y_Cleaned',
       'Global_Rank_Score', 'Research_Productivity_Index',
       'Faculty_Student_Ratio', 'Intl_Student_Percentage',
       'Performance_Index'],
      dtype='object')


In [ ]:
df['University_Name'] = df['Institution Name'].fillna(df['Name'])

df['Country_Final'] = df['Location'].fillna(df['Country'])

df['Publications'] = df['Research Environment']

df['Citations'] = df['Research Quality']

region_map = {
    'india':'Asia',
    'china':'Asia',
    'japan':'Asia',
    'south korea':'Asia',
    'singapore':'Asia',
    'malaysia':'Asia',
    'thailand':'Asia',
    'united states':'North America',
    'canada':'North America',
    'mexico':'North America',
    'united kingdom':'Europe',
    'england':'Europe',
    'france':'Europe',
    'germany':'Europe',
    'italy':'Europe',
    'spain':'Europe',
    'netherlands':'Europe',
    'switzerland':'Europe',
    'sweden':'Europe',
    'norway':'Europe',
    'finland':'Europe',
    'denmark':'Europe',
    'ireland':'Europe',
    'belgium':'Europe',
    'australia':'Oceania',
    'new zealand':'Oceania',
    'brazil':'South America',
    'argentina':'South America',
    'chile':'South America',
    'south africa':'Africa',
    'egypt':'Africa',
    'kenya':'Africa'
}

df['Region'] = df['Country_Final'].map(region_map)
df['Region'] = df['Region'].fillna('Other')

df['Student_Diversity'] = df['Intl_Student_Percentage']

df.to_csv("/content/drive/MyDrive/university_final_dataset.csv", index=False)

In [ ]:
# STEP 1: Clean THE World University Rankings dataset

# Fix Rank -> integer
the['Rank'] = the['Rank'].astype(float).astype(int)

# Fix Student Population -> numeric (remove any commas just in case, then convert)
the['Student Population'] = (
    the['Student Population'].astype(str).str.replace(',', '', regex=False)
)
the['Student Population'] = pd.to_numeric(the['Student Population'], errors='coerce')

# Fix International Students -> numeric % (remove the % sign)
the['International Students'] = (
    the['International Students'].astype(str).str.replace('%', '', regex=False)
)
the['International Students'] = pd.to_numeric(the['International Students'], errors='coerce')

# Fix Female to Male Ratio -> Excel corrupted many values into time format (46:54:00)
# or decimals (0.069444444). We'll rebuild a clean "Female %" numeric column instead.
def clean_female_pct(val):
    if pd.isna(val):
        return None
    val = str(val).strip()
    # Case 1: normal "46 : 54" or "46:54"
    if ':' in val and val.count(':') <= 2:
        parts = val.split(':')
        try:
            female = float(parts[0].strip())
            male = float(parts[1].strip())
            total = female + male
            if total > 0:
                return round((female / total) * 100, 1)
        except ValueError:
            pass
    # Case 2: corrupted decimal (Excel time fraction), e.g. 0.069444444 -> 1:40 -> not usable directly
    return None

the['Female %'] = the['Female to Male Ratio'].apply(clean_female_pct)

# Trim whitespace on Name and Country
the['Name'] = the['Name'].astype(str).str.strip()
the['Country'] = the['Country'].astype(str).str.strip()

# Drop exact duplicate rows if any
the = the.drop_duplicates()

# Quick check
print(the.dtypes)
print(the[['Rank','Student Population','International Students','Female to Male Ratio','Female %']].head(10))
print("\nNulls after cleaning:\n", the.isnull().sum())

Rank                         int64
Name                        object
Country                     object
Student Population         float64
Students to Staff Ratio    float64
International Students     float64
Female to Male Ratio        object
Overall Score              float64
Teaching                   float64
Research Environment       float64
Research Quality           float64
Industry Impact            float64
International Outlook      float64
Year                         int64
Female %                   float64
dtype: object
   Rank  Student Population  International Students Female to Male Ratio  \
0     1              2243.0                    26.0              33 : 67   
1     2             19920.0                    34.0             46:54:00   
2     3             15596.0                    22.0             42:58:00   
3     4             18810.0                    34.0             46:54:00   
4     5             11074.0                    33.0              37 : 63   
5    

In [ ]:
# STEP 2: Clean QS World Rankings dataset

# Trim whitespace from Institution Name
qs['Institution Name'] = qs['Institution Name'].astype(str).str.strip()

# --- Fix 2025 Rank and 2024 Rank ---
# These contain band ranges like "601-610" or "1401+" instead of a single number.
# We'll create a clean numeric rank (using the lower bound of the band) AND
# keep the original text as "Rank Band" so you don't lose that context.

def clean_rank(val):
    if pd.isna(val):
        return None
    val = str(val).strip()
    val = val.replace('+', '')          # "1401+" -> "1401"
    if '-' in val:
        val = val.split('-')[0]         # "601-610" -> "601"
    try:
        return int(val)
    except ValueError:
        return None

qs['2025 Rank Band'] = qs['2025 Rank'].astype(str)   # keep original text
qs['2025 Rank'] = qs['2025 Rank'].apply(clean_rank)

qs['2024 Rank Band'] = qs['2024 Rank'].astype(str)
qs['2024 Rank'] = qs['2024 Rank'].apply(clean_rank)

# --- Fix QS Overall Score: "-" means missing ---
qs['QS Overall Score'] = qs['QS Overall Score'].astype(str).str.replace('-', '', regex=False)
qs['QS Overall Score'] = qs['QS Overall Score'].replace('', None)
qs['QS Overall Score'] = pd.to_numeric(qs['QS Overall Score'], errors='coerce')

# Trim whitespace on Location / Location Full too, just to be safe
qs['Location'] = qs['Location'].astype(str).str.strip()
qs['Location Full'] = qs['Location Full'].astype(str).str.strip()

# Drop exact duplicate rows if any
qs = qs.drop_duplicates()

# Quick check
print(qs.dtypes)
print(qs[['2025 Rank','2025 Rank Band','2024 Rank','2024 Rank Band','QS Overall Score']].head(10))
print("\nNulls after cleaning:\n", qs.isnull().sum())

2025 Rank                           int64
2024 Rank                         float64
Institution Name                   object
Location                           object
Location Full                      object
Size                               object
Academic Reputation               float64
Employer Reputation               float64
Faculty Student                   float64
Citations per Faculty             float64
International Faculty             float64
International Students            float64
International Research Network    float64
Employment Outcomes               float64
Sustainability                    float64
QS Overall Score                  float64
2025 Rank Band                     object
2024 Rank Band                     object
dtype: object
   2025 Rank 2025 Rank Band  2024 Rank 2024 Rank Band  QS Overall Score
0          1              1        1.0              1             100.0
1          2              2        6.0              6              98.5
2          3  

In [ ]:
# STEP 3: Build a standardized name key for matching THE and QS

!pip install unidecode -q
from unidecode import unidecode
import re

def clean_name(name):
    name = unidecode(str(name)).lower().strip()   # remove accents (é -> e, ü -> u)
    name = re.sub(r'\(.*?\)', '', name)            # remove "(MIT)", "(NUS)" etc.
    name = name.replace('&', 'and')
    name = re.sub(r'^the\s+', '', name)            # drop leading "The "
    name = re.sub(r'[^a-z0-9\s]', ' ', name)        # remove commas, periods, hyphens
    name = re.sub(r'\s+', ' ', name).strip()        # collapse extra spaces
    return name

qs['Name_clean'] = qs['Institution Name'].apply(clean_name)
the['Name_clean'] = the['Name'].apply(clean_name)

# Test how many QS institutions find an exact match in THE
qs_names = set(qs['Name_clean'].unique())
the_names = set(the['Name_clean'].unique())
matched = qs_names & the_names

print(f"QS institutions: {len(qs_names)}")
print(f"Exact matches found in THE: {len(matched)}")
print(f"Still unmatched: {len(qs_names - the_names)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 7.8 MB/s eta 0:00:00
QS institutions: 1502
Exact matches found in THE: 1022
Still unmatched: 480


In [ ]:
# STEP 4: Suggest fuzzy matches for the ~480 unmatched QS institutions
# (restricted to same country, so we don't match across countries by accident)

!pip install rapidfuzz -q
from rapidfuzz import process, fuzz

# Align country names between the two datasets (QS and THE spell some differently)
country_map = {
    'China (Mainland)': 'China',
    'Hong Kong SAR': 'Hong Kong',
    'Macau SAR': 'Macao',
    'Russia': 'Russian Federation',
    'Iran, Islamic Republic of': 'Iran',
    'Brunei': 'Brunei Darussalam',
}
qs['Country_mapped'] = qs['Location Full'].replace(country_map)

# Build a lookup of THE institution names, grouped by country
the_by_country = the.groupby('Country')['Name_clean'].apply(lambda x: sorted(set(x))).to_dict()

# Only look at QS rows that didn't get an exact match
unmatched_qs = qs[~qs['Name_clean'].isin(the_names)].copy()

def find_best_match(row):
    candidates = the_by_country.get(row['Country_mapped'], [])
    if not candidates:
        return pd.Series([None, 0])
    result = process.extractOne(row['Name_clean'], candidates, scorer=fuzz.token_sort_ratio)
    return pd.Series([result[0], result[1]]) if result else pd.Series([None, 0])

unmatched_qs[['Best_THE_Match', 'Match_Score']] = unmatched_qs.apply(find_best_match, axis=1)

review = unmatched_qs[['Institution Name', 'Country_mapped', 'Best_THE_Match', 'Match_Score']]
review = review.sort_values('Match_Score', ascending=False).reset_index(drop=True)

review.to_csv('/content/fuzzy_match_review.csv', index=False)
print(f"Generated {len(review)} candidate matches for review")
review.head(30)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.4 MB/s eta 0:00:00
Generated 481 candidate matches for review


,Institution Name,Country_mapped,Best_THE_Match,Match_Score
0,"university politehnica of timisoara, upt",Romania,NaN,0.0
1,universidad autónoma de san luis de potosí,Mexico,NaN,0.0
2,universidad bernardo o'higgins,Chile,NaN,0.0
3,"universidad católica boliviana ""san pablo""",Bolivia,NaN,0.0
4,universidad católica de colombia,Colombia,NaN,0.0
5,universidad católica de la santísima concepció...,Chile,NaN,0.0
6,universidad católica de santiago de guayaquil,Ecuador,NaN,0.0
7,universidad católica de temuco,Chile,NaN,0.0
8,universidad central de chile,Chile,NaN,0.0
9,russian state agrarian university - moscow tim...,Russian Federation,NaN,0.0


In [ ]:
# STEP 5: Apply fuzzy matches, excluding confirmed false positives

# Institutions where the fuzzy match is WRONG (different university despite similar name)
# — confirmed by manual review, mostly city/state naming collisions
manual_rejects = [
    'Northeastern University', 'Southwest University', 'Universidade Federal do Parà - UFPA',
    'Universidade Federal da Paraíba', 'California State University - Los Angeles',
    'Universitas Brawijaya', 'MGIMO University', 'Lanzhou University',
    'Universiti Sains Islam Malaysia', 'Universidade Estadual de Londrina',
    'Universidade Federal do Rio Grande Do Norte', 'Indiana State University',
    'National Research Tomsk Polytechnic University', 'Universidad Austral de Chile',
    'Purdue University', 'Universidade Federal de Pelotas', 'University of Bari',
    'Universidade Federal do Rio de Janeiro', 'Loyola University Chicago',
    'University of Mississippi', 'Universidad Católica de Temuco',
    'Universidade Federal de Goiás', 'Universidad Católica del Norte',
    'Universidade Federal de Santa Maria', 'Universiti Malaysia Pahang',
    'Universidade de São Paulo', 'Dongguk University',
    'Universidad Popular Autónoma del Estado de Puebla (UPAEP)', 'Hohai University',
    'Colorado State University', 'AGH University of Science and Technology',
    'University of Bialystok', 'Northwest Agriculture and Forestry University',
    'Pontifícia Universidade Católica de São Paulo', 'University of Massachusetts Boston',
]

# Accept fuzzy matches scoring >= 85, minus the confirmed rejects
approved = review[(review['Match_Score'] >= 85) & (~review['Institution Name'].isin(manual_rejects))]

print(f"Exact matches (Step 3):        1022")
print(f"Approved fuzzy matches:        {len(approved)}")
print(f"Rejected (confirmed wrong):    {len(manual_rejects)}")
print(f"Total matched to THE:          {1022 + len(approved)} / 1502")

# Build the final name mapping: QS Institution Name -> THE Name_clean
fuzzy_map = dict(zip(approved['Institution Name'], approved['Best_THE_Match']))
print("\nSample approved mappings:")
for k, v in list(fuzzy_map.items())[:10]:
    print(f"  {k}  ->  {v}")

Exact matches (Step 3):        1022
Approved fuzzy matches:        0
Rejected (confirmed wrong):    35
Total matched to THE:          1022 / 1502

Sample approved mappings:


In [ ]:
# STEP 5 (fixed): Rebuild review fresh, then apply approved fuzzy matches
# (rebuilding here avoids any interference from the interactive table widget)

unmatched_qs2 = qs[~qs['Name_clean'].isin(the_names)].copy()
unmatched_qs2[['Best_THE_Match', 'Match_Score']] = unmatched_qs2.apply(find_best_match, axis=1)
review = unmatched_qs2[['Institution Name', 'Country_mapped', 'Best_THE_Match', 'Match_Score']]

print("Total unmatched QS rows:", len(review))
print("Match_Score dtype:", review['Match_Score'].dtype)
print("Rows scoring >= 85:", (review['Match_Score'] >= 85).sum())

manual_rejects = [
    'Northeastern University', 'Southwest University', 'Universidade Federal do Parà - UFPA',
    'Universidade Federal da Paraíba', 'California State University - Los Angeles',
    'Universitas Brawijaya', 'MGIMO University', 'Lanzhou University',
    'Universiti Sains Islam Malaysia', 'Universidade Estadual de Londrina',
    'Universidade Federal do Rio Grande Do Norte', 'Indiana State University',
    'National Research Tomsk Polytechnic University', 'Universidad Austral de Chile',
    'Purdue University', 'Universidade Federal de Pelotas', 'University of Bari',
    'Universidade Federal do Rio de Janeiro', 'Loyola University Chicago',
    'University of Mississippi', 'Universidad Católica de Temuco',
    'Universidade Federal de Goiás', 'Universidad Católica del Norte',
    'Universidade Federal de Santa Maria', 'Universiti Malaysia Pahang',
    'Universidade de São Paulo', 'Dongguk University',
    'Universidad Popular Autónoma del Estado de Puebla (UPAEP)', 'Hohai University',
    'Colorado State University', 'AGH University of Science and Technology',
    'University of Bialystok', 'Northwest Agriculture and Forestry University',
    'Pontifícia Universidade Católica de São Paulo', 'University of Massachusetts Boston',
]

approved = review[(review['Match_Score'] >= 85) & (~review['Institution Name'].isin(manual_rejects))]

print(f"\nExact matches (Step 3):        1022")
print(f"Approved fuzzy matches:        {len(approved)}")
print(f"Rejected (confirmed wrong):    {len(manual_rejects)}")
print(f"Total matched to THE:          {1022 + len(approved)} / 1502")

fuzzy_map = dict(zip(approved['Institution Name'], approved['Best_THE_Match']))
print("\nSample approved mappings:")
for k, v in list(fuzzy_map.items())[:10]:
    print(f"  {k}  ->  {v}")

Total unmatched QS rows: 481
Match_Score dtype: float64
Rows scoring >= 85: 0

Exact matches (Step 3):        1022
Approved fuzzy matches:        0
Rejected (confirmed wrong):    35
Total matched to THE:          1022 / 1502

Sample approved mappings:


In [ ]:
# STEP 5 (fully self-contained — rebuilds everything, ignore any earlier versions)
import pandas as pd, re
from unidecode import unidecode
from rapidfuzz import process, fuzz

# Reload both files fresh to guarantee a clean state
the = pd.read_csv('/content/drive/MyDrive/The World University Ranking 2016-2026.csv')
qs = pd.read_csv('/content/drive/MyDrive/Qs World Ranking 2025.csv')

def clean_name(name):
    name = unidecode(str(name)).lower().strip()
    name = re.sub(r'\(.*?\)', '', name)
    name = name.replace('&', 'and')
    name = re.sub(r'^the\s+', '', name)
    name = re.sub(r'[^a-z0-9\s]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

qs['Name_clean'] = qs['Institution Name'].apply(clean_name)
the['Name_clean'] = the['Name'].apply(clean_name)
the_names = set(the['Name_clean'].unique())

country_map = {'China (Mainland)': 'China','Hong Kong SAR': 'Hong Kong','Macau SAR': 'Macao',
    'Russia': 'Russian Federation','Iran, Islamic Republic of': 'Iran','Brunei': 'Brunei Darussalam'}
qs['Country_mapped'] = qs['Location Full'].replace(country_map)
the_by_country = the.groupby('Country')['Name_clean'].apply(lambda x: sorted(set(x))).to_dict()

def find_best_match(row):
    candidates = the_by_country.get(row['Country_mapped'], [])
    if not candidates:
        return pd.Series([None, 0])
    result = process.extractOne(row['Name_clean'], candidates, scorer=fuzz.token_sort_ratio)
    return pd.Series([result[0], result[1]]) if result else pd.Series([None, 0])

# Sanity check on one known case before trusting the full run
test = find_best_match(pd.Series({'Country_mapped': 'United States'}).combine_first(
    pd.Series({'Name_clean': clean_name('Purdue University')})))
print("Sanity check (Purdue, should score ~87):", test.tolist())

unmatched_qs2 = qs[~qs['Name_clean'].isin(the_names)].copy()
unmatched_qs2[['Best_THE_Match', 'Match_Score']] = unmatched_qs2.apply(find_best_match, axis=1)
review = unmatched_qs2[['Institution Name', 'Country_mapped', 'Best_THE_Match', 'Match_Score']]

print("Total unmatched QS rows:", len(review))
print("Rows scoring >= 85:", (review['Match_Score'] >= 85).sum())

manual_rejects = [
    'Northeastern University', 'Southwest University', 'Universidade Federal do Parà - UFPA',
    'Universidade Federal da Paraíba', 'California State University - Los Angeles',
    'Universitas Brawijaya', 'MGIMO University', 'Lanzhou University',
    'Universiti Sains Islam Malaysia', 'Universidade Estadual de Londrina',
    'Universidade Federal do Rio Grande Do Norte', 'Indiana State University',
    'National Research Tomsk Polytechnic University', 'Universidad Austral de Chile',
    'Purdue University', 'Universidade Federal de Pelotas', 'University of Bari',
    'Universidade Federal do Rio de Janeiro', 'Loyola University Chicago',
    'University of Mississippi', 'Universidad Católica de Temuco',
    'Universidade Federal de Goiás', 'Universidad Católica del Norte',
    'Universidade Federal de Santa Maria', 'Universiti Malaysia Pahang',
    'Universidade de São Paulo', 'Dongguk University',
    'Universidad Popular Autónoma del Estado de Puebla (UPAEP)', 'Hohai University',
    'Colorado State University', 'AGH University of Science and Technology',
    'University of Bialystok', 'Northwest Agriculture and Forestry University',
    'Pontifícia Universidade Católica de São Paulo', 'University of Massachusetts Boston',
]

approved = review[(review['Match_Score'] >= 85) & (~review['Institution Name'].isin(manual_rejects))]

print(f"\nExact matches (Step 3):        1022")
print(f"Approved fuzzy matches:        {len(approved)}")
print(f"Rejected (confirmed wrong):    {len(manual_rejects)}")
print(f"Total matched to THE:          {1022 + len(approved)} / 1502")

fuzzy_map = dict(zip(approved['Institution Name'], approved['Best_THE_Match']))

Sanity check (Purdue, should score ~87): ['duke university', 87.5]
Total unmatched QS rows: 481
Rows scoring >= 85: 149

Exact matches (Step 3):        1022
Approved fuzzy matches:        117
Rejected (confirmed wrong):    35
Total matched to THE:          1139 / 1502


In [ ]:
# STEP 6: Merge THE (all years) with QS 2025 data

# Give every QS row its final "key" for joining to THE
def get_match_key(row):
    if row['Name_clean'] in the_names:
        return row['Name_clean']
    elif row['Institution Name'] in fuzzy_map:
        return fuzzy_map[row['Institution Name']]
    return None

qs['THE_Match_Key'] = qs.apply(get_match_key, axis=1)
print("QS rows with a usable match key:", qs['THE_Match_Key'].notna().sum(), "/", len(qs))

# Select QS columns and rename with a "QS " prefix, so once merged it's always
# clear which ranking system a column belongs to
qs_export = qs[[
    'THE_Match_Key', 'Institution Name', 'Location Full', 'Size',
    '2025 Rank', '2025 Rank Band', '2024 Rank', '2024 Rank Band',
    'Academic Reputation', 'Employer Reputation', 'Faculty Student',
    'Citations per Faculty', 'International Faculty', 'International Students',
    'International Research Network', 'Employment Outcomes', 'Sustainability',
    'QS Overall Score'
]].copy()

qs_export = qs_export.rename(columns={
    'Institution Name': 'QS Institution Name', 'Location Full': 'QS Country',
    'Size': 'QS Size', '2025 Rank': 'QS 2025 Rank', '2025 Rank Band': 'QS 2025 Rank Band',
    '2024 Rank': 'QS 2024 Rank', '2024 Rank Band': 'QS 2024 Rank Band',
    'Academic Reputation': 'QS Academic Reputation', 'Employer Reputation': 'QS Employer Reputation',
    'Faculty Student': 'QS Faculty Student Ratio', 'Citations per Faculty': 'QS Citations per Faculty',
    'International Faculty': 'QS International Faculty Score',
    'International Students': 'QS International Students Score',
    'International Research Network': 'QS International Research Network',
    'Employment Outcomes': 'QS Employment Outcomes', 'Sustainability': 'QS Sustainability',
})

# Safety check: if two QS institutions somehow mapped to the same THE key,
# keep only the better-ranked one so the merge doesn't duplicate THE rows
qs_export = qs_export.sort_values('QS 2025 Rank').drop_duplicates(subset='THE_Match_Key', keep='first')

# Prefix THE's own ranking columns for the same clarity
the_export = the.rename(columns={
    'Rank': 'THE Rank', 'Students to Staff Ratio': 'THE Students to Staff Ratio',
    'Overall Score': 'THE Overall Score', 'Teaching': 'THE Teaching',
    'Research Environment': 'THE Research Environment', 'Research Quality': 'THE Research Quality',
    'Industry Impact': 'THE Industry Impact', 'International Outlook': 'THE International Outlook',
})

# The actual merge: every THE university-year row, left-joined to matching QS data
merged = the_export.merge(qs_export, left_on='Name_clean', right_on='THE_Match_Key', how='left')

print("\nOriginal THE row count:", len(the_export))
print("Merged row count:      ", len(merged))
print("(these two numbers should match — if not, something duplicated)")
print("\nRows with QS data attached:", merged['QS Institution Name'].notna().sum())
print("Rows with no QS match:     ", merged['QS Institution Name'].isna().sum())
merged.head(10)

QS rows with a usable match key: 1139 / 1503


KeyError: "['2025 Rank Band', '2024 Rank Band'] not in index"

In [ ]:
# STEP 6 (COMPLETE PIPELINE — run this as one cell, replaces everything before it)
!pip install unidecode rapidfuzz -q
import pandas as pd, re
from unidecode import unidecode
from rapidfuzz import process, fuzz

# ---- Load raw (use YOUR actual Drive paths from your very first cell) ----
the = pd.read_csv('/content/drive/MyDrive/The World University Ranking 2016-2026.csv')
qs = pd.read_csv('/content/drive/MyDrive/Qs World Ranking 2025.csv')

# ---- Clean THE ----
the['Rank'] = the['Rank'].astype(float).astype(int)
the['Student Population'] = pd.to_numeric(the['Student Population'].astype(str).str.replace(',','',regex=False), errors='coerce')
the['International Students'] = pd.to_numeric(the['International Students'].astype(str).str.replace('%','',regex=False), errors='coerce')

def clean_female_pct(val):
    if pd.isna(val): return None
    val = str(val).strip()
    if ':' in val and val.count(':') <= 2:
        parts = val.split(':')
        try:
            f, m = float(parts[0].strip()), float(parts[1].strip())
            t = f + m
            if t > 0: return round((f/t)*100, 1)
        except ValueError: pass
    return None

the['Female %'] = the['Female to Male Ratio'].apply(clean_female_pct)
the['Name'] = the['Name'].astype(str).str.strip()
the['Country'] = the['Country'].astype(str).str.strip()
the = the.drop_duplicates()

# ---- Clean QS ----
qs['Institution Name'] = qs['Institution Name'].astype(str).str.strip()

def clean_rank(val):
    if pd.isna(val): return None
    val = str(val).strip().replace('+', '')
    if '-' in val: val = val.split('-')[0]
    try: return int(val)
    except ValueError: return None

qs['2025 Rank Band'] = qs['2025 Rank'].astype(str)
qs['2025 Rank'] = qs['2025 Rank'].apply(clean_rank)
qs['2024 Rank Band'] = qs['2024 Rank'].astype(str)
qs['2024 Rank'] = qs['2024 Rank'].apply(clean_rank)
qs['QS Overall Score'] = qs['QS Overall Score'].astype(str).str.replace('-', '', regex=False).replace('', None)
qs['QS Overall Score'] = pd.to_numeric(qs['QS Overall Score'], errors='coerce')
qs['Location'] = qs['Location'].astype(str).str.strip()
qs

,2025 Rank,2024 Rank,Institution Name,Location,Location Full,Size,Academic Reputation,Employer Reputation,Faculty Student,Citations per Faculty,International Faculty,International Students,International Research Network,Employment Outcomes,Sustainability,QS Overall Score,2025 Rank Band,2024 Rank Band
0,1,1.0,Massachusetts Institute of Technology (MIT),US,United States,M,100.0,100.0,100.0,100.0,99.3,86.8,96.0,100.0,99.0,100.0,1,1
1,2,6.0,Imperial College London,UK,United Kingdom,L,98.5,99.5,98.2,93.9,100.0,99.6,97.4,93.4,99.7,98.5,2,6
2,3,3.0,University of Oxford,UK,United Kingdom,L,100.0,100.0,100.0,84.8,98.1,97.7,100.0,100.0,85.0,96.9,3,3
3,4,4.0,Harvard University,US,United States,L,100.0,100.0,96.3,100.0,74.1,69.0,99.6,100.0,84.4,96.8,4,4
4,5,2.0,University of Cambridge,UK,United Kingdom,L,100.0,100.0,100.0,84.6,100.0,94.8,99.3,100.0,84.8,96.7,5,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1498,1401,1201.0,University of Montana Missoula,US,United States,M,3.0,2.2,10.6,6.1,1.3,1.9,6.5,3.1,1.0,NaN,1401+,1201-1400
1499,1401,1401.0,University of Oradea,RO,Romania,L,5.6,2.2,4.0,1.9,1.5,5.2,34.5,6.2,2.3,NaN,1401+,1401+
1500,1401,1201.0,University of San Carlos,PH,Philippines,M,7.2,9.4,3.3,1.8,2.1,2.1,6.4,9.6,1.0,NaN,1401+,1201-1400
1501,1401,1401.0,"University Politehnica of Timisoara, UPT",RO,Romania,L,4.1,4.2,7.2,3.9,1.4,2.5,18.6,3.9,1.1,NaN,1401+,1401+


In [ ]:
# STEP 6 (COMPLETE PIPELINE — run this as one cell, replaces everything before it)
!pip install unidecode rapidfuzz -q
import pandas as pd, re
from unidecode import unidecode
from rapidfuzz import process, fuzz

# ---- Load raw (use YOUR actual Drive paths from your very first cell) ----
the = pd.read_csv('/content/drive/MyDrive/The World University Ranking 2016-2026.csv')
qs = pd.read_csv('/content/drive/MyDrive/Qs World Ranking 2025.csv')

# ---- Clean THE ----
the['Rank'] = the['Rank'].astype(float).astype(int)
the['Student Population'] = pd.to_numeric(the['Student Population'].astype(str).str.replace(',','',regex=False), errors='coerce')
the['International Students'] = pd.to_numeric(the['International Students'].astype(str).str.replace('%','',regex=False), errors='coerce')

def clean_female_pct(val):
    if pd.isna(val): return None
    val = str(val).strip()
    if ':' in val and val.count(':') <= 2:
        parts = val.split(':')
        try:
            f, m = float(parts[0].strip()), float(parts[1].strip())
            t = f + m
            if t > 0: return round((f/t)*100, 1)
        except ValueError: pass
    return None

the['Female %'] = the['Female to Male Ratio'].apply(clean_female_pct)
the['Name'] = the['Name'].astype(str).str.strip()
the['Country'] = the['Country'].astype(str).str.strip()
the = the.drop_duplicates()

# ---- Clean QS ----
qs['Institution Name'] = qs['Institution Name'].astype(str).str.strip()

def clean_rank(val):
    if pd.isna(val): return None
    val = str(val).strip().replace('+', '')
    if '-' in val: val = val.split('-')[0]
    try: return int(val)
    except ValueError: return None

qs['2025 Rank Band'] = qs['2025 Rank'].astype(str)
qs['2025 Rank'] = qs['2025 Rank'].apply(clean_rank)
qs['2024 Rank Band'] = qs['2024 Rank'].astype(str)
qs['2024 Rank'] = qs['2024 Rank'].apply(clean_rank)
qs['QS Overall Score'] = qs['QS Overall Score'].astype(str).str.replace('-', '', regex=False).replace('', None)
qs['QS Overall Score'] = pd.to_numeric(qs['QS Overall Score'], errors='coerce')
qs['Location'] = qs['Location'].astype(str).str.strip()
qs['Location Full'] = qs['Location Full'].astype(str).str.strip()
qs = qs.drop_duplicates()

# ---- Build standardized name keys ----
def clean_name(name):
    name = unidecode(str(name)).lower().strip()
    name = re.sub(r'\(.*?\)', '', name)
    name = name.replace('&', 'and')
    name = re.sub(r'^the\s+', '', name)
    name = re.sub(r'[^a-z0-9\s]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

qs['Name_clean'] = qs['Institution Name'].apply(clean_name)
the['Name_clean'] = the['Name'].apply(clean_name)
the_names = set(the['Name_clean'].unique())

country_map = {'China (Mainland)': 'China', 'Hong Kong SAR': 'Hong Kong', 'Macau SAR': 'Macao',
    'Russia': 'Russian Federation', 'Iran, Islamic Republic of': 'Iran', 'Brunei': 'Brunei Darussalam'}
qs['Country_mapped'] = qs['Location Full'].replace(country_map)
the_by_country = the.groupby('Country')['Name_clean'].apply(lambda x: sorted(set(x))).to_dict()

def find_best_match(row):
    candidates = the_by_country.get(row['Country_mapped'], [])
    if not candidates: return pd.Series([None, 0])
    r = process.extractOne(row['Name_clean'], candidates, scorer=fuzz.token_sort_ratio)
    return pd.Series([r[0], r[1]]) if r else pd.Series([None, 0])

unmatched_qs = qs[~qs['Name_clean'].isin(the_names)].copy()
unmatched_qs[['Best_THE_Match', 'Match_Score']] = unmatched_qs.apply(find_best_match, axis=1)
review = unmatched_qs[['Institution Name', 'Country_mapped', 'Best_THE_Match', 'Match_Score']]

manual_rejects = ['Northeastern University','Southwest University','Universidade Federal do Parà - UFPA',
 'Universidade Federal da Paraíba','California State University - Los Angeles','Universitas Brawijaya',
 'MGIMO University','Lanzhou University','Universiti Sains Islam Malaysia','Universidade Estadual de Londrina',
 'Universidade Federal do Rio Grande Do Norte','Indiana State University','National Research Tomsk Polytechnic University',
 'Universidad Austral de Chile','Purdue University','Universidade Federal de Pelotas','University of Bari',
 'Universidade Federal do Rio de Janeiro','Loyola University Chicago','University of Mississippi',
 'Universidad Católica de Temuco','Universidade Federal de Goiás','Universidad Católica del Norte',
 'Universidade Federal de Santa Maria','Universiti Malaysia Pahang','Universidade de São Paulo','Dongguk University',
 'Universidad Popular Autónoma del Estado de Puebla (UPAEP)','Hohai University','Colorado State University',
 'AGH University of Science and Technology','University of Bialystok','Northwest Agriculture and Forestry University',
 'Pontifícia Universidade Católica de São Paulo','University of Massachusetts Boston']

approved = review[(review['Match_Score'] >= 85) & (~review['Institution Name'].isin(manual_rejects))]
fuzzy_map = dict(zip(approved['Institution Name'], approved['Best_THE_Match']))

def get_match_key(row):
    if row['Name_clean'] in the_names: return row['Name_clean']
    elif row['Institution Name'] in fuzzy_map: return fuzzy_map[row['Institution Name']]
    return None

qs['THE_Match_Key'] = qs.apply(get_match_key, axis=1)
print("QS rows with a usable match key:", qs['THE_Match_Key'].notna().sum(), "/", len(qs))

# ---- Prep QS columns for merge ----
qs_cols = ['THE_Match_Key','Institution Name','Location Full','Size','2025 Rank','2025 Rank Band',
 '2024 Rank','2024 Rank Band','Academic Reputation','Employer Reputation','Faculty Student',
 'Citations per Faculty','International Faculty','International Students','International Research Network',
 'Employment Outcomes','Sustainability','QS Overall Score']
qs_export = qs[qs_cols].copy()
qs_export = qs_export.rename(columns={'Institution Name':'QS Institution Name','Location Full':'QS Country',
 'Size':'QS Size','2025 Rank':'QS 2025 Rank','2025 Rank Band':'QS 2025 Rank Band','2024 Rank':'QS 2024 Rank',
 '2024 Rank Band':'QS 2024 Rank Band','Academic Reputation':'QS Academic Reputation',
 'Employer Reputation':'QS Employer Reputation','Faculty Student':'QS Faculty Student Ratio',
 'Citations per Faculty':'QS Citations per Faculty','International Faculty':'QS International Faculty Score',
 'International Students':'QS International Students Score',
 'International Research Network':'QS International Research Network',
 'Employment Outcomes':'QS Employment Outcomes','Sustainability':'QS Sustainability'})

# If two QS rows map to the same THE key, keep only the higher-ranked one
qs_export = qs_export.sort_values('QS 2025 Rank').drop_duplicates(subset='THE_Match_Key', keep='first')

the_export = the.rename(columns={'Rank': 'THE Rank'})

# ---- THE MERGE ----
merged = the_export.merge(qs_export, left_on='Name_clean', right_on='THE_Match_Key', how='left')

print("\nOriginal THE row count:", len(the_export))
print("Merged row count:      ", len(merged))
print("Rows with QS data attached:", merged['QS Institution Name'].notna().sum())
merged.head(10)

QS rows with a usable match key: 1136 / 1503

Original THE row count: 16713
Merged row count:       16713
Rows with QS data attached: 9868


,THE Rank,Name,Country,Student Population,Students to Staff Ratio,International Students,Female to Male Ratio,Overall Score,Teaching,Research Environment,...,QS Academic Reputation,QS Employer Reputation,QS Faculty Student Ratio,QS Citations per Faculty,QS International Faculty Score,QS International Students Score,QS International Research Network,QS Employment Outcomes,QS Sustainability,QS Overall Score
0,1,California Institute of Technology,United States,2243.0,6.9,26.0,33 : 67,95.2,95.6,97.6,...,96.5,95.3,100.0,100.0,100.0,79.8,65.5,31.0,62.5,90.9
1,2,University of Oxford,United Kingdom,19920.0,11.6,34.0,46:54:00,94.2,86.5,98.9,...,100.0,100.0,100.0,84.8,98.1,97.7,100.0,100.0,85.0,96.9
2,3,Stanford University,United States,15596.0,7.8,22.0,42:58:00,93.9,92.5,96.2,...,100.0,100.0,100.0,99.0,70.3,60.8,96.8,100.0,81.2,96.1
3,4,University of Cambridge,United Kingdom,18810.0,11.8,34.0,46:54:00,92.8,88.2,96.7,...,100.0,100.0,100.0,84.6,100.0,94.8,99.3,100.0,84.8,96.7
4,5,Massachusetts Institute of Technology,United States,11074.0,9.0,33.0,37 : 63,92.0,89.4,88.6,...,100.0,100.0,100.0,100.0,99.3,86.8,96.0,100.0,99.0,100.0
5,6,Harvard University,United States,20152.0,8.9,25.0,NaN,91.6,83.6,99.0,...,100.0,100.0,96.3,100.0,74.1,69.0,99.6,100.0,84.4,96.8
6,7,Princeton University,United States,7929.0,8.4,27.0,45:55:00,90.1,85.1,91.9,...,99.8,98.3,57.0,100.0,9.6,56.6,78.3,95.7,51.5,85.5
7,8,Imperial College London,United Kingdom,15060.0,11.7,51.0,37 : 63,89.1,83.3,88.5,...,98.5,99.5,98.2,93.9,100.0,99.6,97.4,93.4,99.7,98.5
8,9,ETH Zurich,Switzerland,18178.0,14.7,37.0,31 : 69,88.3,77.0,95.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,The University of Chicago,United States,14221.0,6.9,20.0,42:58:00,87.9,85.7,88.9,...,99.1,96.4,94.2,60.8,79.0,87.0,85.1,99.1,46.9,86.2


In [ ]:
# STEP 7: Final cleanup and export for Tableau

# Drop helper columns only used for matching, not needed in the final dataset
final = merged.drop(columns=['Name_clean', 'THE_Match_Key'])

# Reorder so identity/ranking columns come first (easier to work with in Tableau)
front_cols = ['Year', 'Name', 'Country', 'THE Rank', 'QS 2025 Rank', 'QS 2024 Rank']
other_cols = [c for c in final.columns if c not in front_cols]
final = final[front_cols + other_cols]

# Final checks before export
print("Final shape:", final.shape)
print("\nColumn list:")
print(final.columns.tolist())
print("\nDuplicate rows:", final.duplicated().sum())
print("\nNulls per column:")
print(final.isnull().sum())

# Export
final.to_csv('/content/drive/MyDrive/EduVision_DV_Merged_Clean.csv', index=False)
print("\nSaved to Google Drive: EduVision_DV_Merged_Clean.csv")

# Also save a copy locally in the Colab session for quick download
final.to_csv('/content/EduVision_DV_Merged_Clean.csv', index=False)

Final shape: (16713, 32)

Column list:
['Year', 'Name', 'Country', 'THE Rank', 'QS 2025 Rank', 'QS 2024 Rank', 'Student Population', 'Students to Staff Ratio', 'International Students', 'Female to Male Ratio', 'Overall Score', 'Teaching', 'Research Environment', 'Research Quality', 'Industry Impact', 'International Outlook', 'Female %', 'QS Institution Name', 'QS Country', 'QS Size', 'QS 2025 Rank Band', 'QS 2024 Rank Band', 'QS Academic Reputation', 'QS Employer Reputation', 'QS Faculty Student Ratio', 'QS Citations per Faculty', 'QS International Faculty Score', 'QS International Students Score', 'QS International Research Network', 'QS Employment Outcomes', 'QS Sustainability', 'QS Overall Score']

Duplicate rows: 0

Nulls per column:
Year                                     0
Name                                     0
Country                                  0
THE Rank                                 0
QS 2025 Rank                          6845
QS 2024 Rank                         